# ATML PA3 Task 2 — Run All
Select **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** and approve the normal Google Drive mount.

This notebook runs the full 196-token, 4,500-update protocol. Re-running automatically recovers compatible checkpoints and reuses validated caches. Results are saved under `/content/drive/MyDrive/ATML_PA3_Task2/`.

**Verification:** 29 CPU tests, real official-data selection, and pretrained SigLIP/Qwen CPU integration passed. T4 memory, FP16 CUDA recovery, full training, and five-hour feasibility are **not yet verified**. The launcher measures and hard-gates these before training. No browser execution was used.

The default five-hour **session budget** includes a 15-minute safety reserve; it is not a claim that the entire experiment finishes in five hours. Run All in the next session resumes a safe stop.

Implementation commit: `11e166e110cf35e3f08e5ce087b69554f3b1634d`. Task 1 is untouched. No report prose is generated.


In [1]:
# Immutable implementation: notebook edits do not silently change the training source.
from pathlib import Path
import os
import subprocess
import sys
from google.colab import drive

SOURCE_COMMIT = "11e166e110cf35e3f08e5ce087b69554f3b1634d"
REPOSITORY = "https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning.git"
drive.mount("/content/drive")
source = Path("/content/atml_task2_source") / SOURCE_COMMIT
source.mkdir(parents=True, exist_ok=True)
def git(*args):
    return subprocess.check_output(["git", "-C", str(source), *args], text=True).strip()
if not (source / ".git").exists():
    git("init")
    git("remote", "add", "origin", REPOSITORY)
try:
    checked_out = git("rev-parse", "HEAD")
except subprocess.CalledProcessError:
    checked_out = None
if checked_out != SOURCE_COMMIT:
    git("fetch", "--depth", "1", "origin", SOURCE_COMMIT)
    git("checkout", "--detach", SOURCE_COMMIT)
assert git("rev-parse", "HEAD") == SOURCE_COMMIT
if git("status", "--porcelain"):
    raise RuntimeError("Pinned source has local edits; refusing to run a modified experiment.")
workdir = source / "Assignment-3"
env = dict(os.environ, TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS="2", HF_HUB_DISABLE_TELEMETRY="1")
print("Task 2 source:", SOURCE_COMMIT, flush=True)
subprocess.run([sys.executable, "-m", "task2.bootstrap"], cwd=workdir, env=env, check=True)
# Real CUDA gates run first. Failure stops here; no silent protocol fallback.
# On reconnect, Run All automatically resumes the latest compatible verified checkpoint.
subprocess.run([sys.executable, "-u", "-m", "task2.run"], cwd=workdir, env=env, check=True)


Mounted at /content/drive
Task 2 source: 11e166e110cf35e3f08e5ce087b69554f3b1634d


CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'task2.run'], returncode=0)

In [3]:
from pathlib import Path
import subprocess, sys

drive = Path("/content/drive/MyDrive")
source = Path(
    "/content/atml_task2_source/"
    "11e166e110cf35e3f08e5ce087b69554f3b1634d/"
    "Assignment-3"
)

print("=== GOOGLE DRIVE ===")
print("Drive mounted:", drive.exists())
print("Task 2 folder:", (drive / "ATML_PA3_Task2").exists())

print("\n=== SOURCE CODE ===")
print("Source exists:", source.exists())
print("Runner exists:", (source / "task2/run.py").exists())

if source.exists():
    result = subprocess.run(
        [sys.executable, "-c",
         "import task2.run; print(task2.run.__file__)"],
        cwd=source,
        capture_output=True,
        text=True
    )
    print("Module check:", result.returncode)
    print("Module path:", result.stdout.strip())
    if result.stderr:
        print("Error:", result.stderr[-1500:])

print("\n=== SAVED RESULTS ===")
if drive.exists():
    matches = list(drive.glob("**/final_verification.json"))
    print("Status files found:", len(matches))
    for p in matches[:10]:
        print(p)

print("\n=== TASK 2 DRIVE CONTENTS ===")
folder = drive / "ATML_PA3_Task2"
if folder.exists():
    for p in list(folder.iterdir())[:20]:
        print(p.name, "DIR" if p.is_dir() else "FILE")
else:
    print("Task 2 Drive folder does not exist.")

AssertionError: Kaggle Persistence/finalizer missing; STOP.

In [4]:
from google.colab import drive
from pathlib import Path
import json

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive/ATML_PA3_Task2")

print("Task 2 Drive folder exists:", root.exists())

if root.exists():
    for path in sorted(root.glob("runs/*/final_verification.json")):
        print("\nRun:", path.parent.name)
        print(json.dumps(json.loads(path.read_text()), indent=2)[:1500])

        checkpoint = path.parent / "checkpoints/latest.json"
        if checkpoint.exists():
            state = json.loads(checkpoint.read_text())
            print("Saved training progress:", state["update"], "/ 4500")
else:
    print("No Task 2 folder found in this Google Drive.")

Mounted at /content/drive
Task 2 Drive folder exists: True

Run: 5db9313b93ea20612b4c
{
  "artifacts": {
    "calibration.json": "dd7d9087a43dc0f88346c6f258250bf26c8f71db5c81a9b9b1cd347bdb416510",
    "cuda-stage-1.json": "780bbfceb7495c23806fbfee7cd522337d407f29885ddc1ad6f402cd4753b7fc",
    "cuda-stage-2.json": "18f5348992e3ff5b176cd52fe51f512ddf5bec88d3406ef39f12d26ea3c3ee7e",
    "cuda-stage-3.json": "c6d60f82d650b050e8fd32561157a4d637ca94e5f7f8a79a639bf79005f7430a",
    "dev-0000.json": "d541b0cdf45d5ed34694d9473f638e9dd942ad1bcfe26fff61bd9a863cff78aa",
    "dev-0500.json": "d6e197d36630aadaf823726fe5801abc22816047f1334150d0aa380f5be4c468",
    "dev-1000.json": "a142c7f13bca3190f2674aa5b97ce840d75e29e3d744f58ad015cbbb1b326474",
    "dev-1500.json": "fba3d8ad6a4639cd16e01e6a9316c6050793a7e329cbd77c243cc544e81babb0",
    "dev-2000.json": "67f73794ba0d20d972728231b66967b390142c27a9b4bd77d223e78dda67f90c",
    "dev-2250.json": "8de58817aa080f3ac7a866b70b972c96322d6f0d1bc43bdeaa6b1d206

In [5]:
from pathlib import Path
import json

p = Path(
    "/content/drive/MyDrive/ATML_PA3_Task2/"
    "runs/5db9313b93ea20612b4c/final_verification.json"
)

result = json.loads(p.read_text())

print("Status:", result.get("status"))
print("Fully completed:", result.get("fully_completed"))
print("Main updates:", result.get("main_updates"))
print("Artifacts saved:", len(result.get("artifacts", {})))

Status: fully completed
Fully completed: True
Main updates: 4500
Artifacts saved: 50
